# ConceptGraphs results: semantic map quality, geometry, and graph cases

This notebook reports saved pipeline results and reconstructs an **observed GT reference** from OSMa semantic masks, depth, calibration, and the exact staged camera poses. It does not run detection, mapping, graph construction, or model inference.

The reference covers the selected frames, not a complete room mesh. OSMa semantic masks are category annotations, not physical-object instance IDs; no ground-truth relationship graph is provided here. Consequently **mAcc/f-mIoU are semantic labeling metrics, not object-detection AP or relationship correctness**.

Run this notebook with the **cleanjupyter Windows kernel**. CPU evaluation is dispatched to the existing Ubuntu mapping environment, which already has SciPy and the map-pickle dependencies. Report files go under `artifacts/reports/`; existing experiments are read only.


In [ ]:
from pathlib import Path
from datetime import datetime
import base64, html, importlib, json, os, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "code/midterm_report/lora_baseline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the Senior Capstone project.")
sys.path.insert(0, str(ROOT / "code/midterm_report"))
from conceptgraphs_support import conceptgraphs_bridge as cg
from conceptgraphs_support import results_report as rr
from conceptgraphs_support import image_evidence as ie
importlib.reload(rr)
importlib.reload(ie)
palette = rr.palette


## 1. Report parameters

Use the same frame set and foreground exclusions for all model conditions. Labels are matched by normalized exact category name. Optional aliases must be explicit and fixed before comparing results; unmapped names are reported rather than silently corrected from GT.


In [ ]:
EXPERIMENT_NAME = "pilot_local_01"
SCENE = "hm3d/00803-k1cupFYWXJ6"
WSL_DISTRO = "Ubuntu"
RUN_EVALUATION = True
# Set this to an existing report folder and RUN_EVALUATION=False to reopen saved results.
REPORT_DIR = None
PIXEL_STRIDE = 4
VOXEL_SIZE_M = 0.025
MAX_DEPTH_M = 10.0
SEMANTIC_ASSOCIATION_M = 0.05
GEOMETRY_THRESHOLDS_M = [0.05, 0.10]
EXCLUDED_CLASSES = ["other", "unknown", "unlabeled", "background", "wall", "floor", "ceiling"]
LABEL_ALIASES = {}  # Example only: {"stairs": "stair"}, if "stair" is in this scene's dictionary.
DISPLAY_VARIANT = "qwen_adapted"
NODE_FRAME_LIMIT = 6
NODE_CROP_PREVIEWS = 12
REPORT_BASE = ROOT / "artifacts/reports" / EXPERIMENT_NAME / SCENE.replace("/", "__")


## 2. Build the observed reference and evaluate saved maps

The reference back-projects depth using the camera intrinsics, then applies the **staged camera-to-world poses**. This upstream mapper uses its original/source-world poses; no first-frame normalization or fitted ICP alignment is applied.

Both clouds use a deterministic sample per 2.5 cm voxel. Image stride and a 10 m depth limit bound processing. The reference view includes background surfaces; quantitative scores use the declared foreground classes. Predicted object-map geometry is compared against that foreground reference, so extraneous predicted surfaces are penalized by geometric precision.

- **mAcc:** mean class recall over GT-supported classes.
- **f-mIoU:** class IoU weighted by reference voxel frequency. Also report unweighted mIoU and point accuracy.
- **Unknown/uncovered:** predictions outside the vocabulary or farther than the semantic association threshold count as errors, not ignored points.
- **Geometry precision/recall/F-score at 5 cm and 10 cm:** surface proximity and coverage; these are point-cloud metrics, not object-instance detection precision/recall.
- **Directional mean distances:** predicted-to-reference (accuracy) and reference-to-predicted (completeness), in metres.
- **Chamfer L1:** sum of the two unsquared directional mean distances, in metres (the factor-of-two convention is explicit).

These are an observed-frame evaluation protocol, not a reproduction of the original Replica evaluation. Since reference and prediction use the same depth and supplied poses, geometry scores measure surface preservation/coverage rather than independent SLAM pose accuracy.

Definitions: [ConceptGraphs semantic metrics](https://github.com/concept-graphs/concept-graphs/blob/main/conceptgraph/utils/eval.py), [Open3D point-cloud metrics](https://www.open3d.org/docs/release/python_api/open3d.t.geometry.PointCloud.html#open3d.t.geometry.PointCloud.compute_metrics).


In [ ]:
if RUN_EVALUATION:
    REPORT_DIR = REPORT_BASE / f"report-{datetime.now():%Y%m%d-%H%M%S-%f}"
    REPORT_DIR.mkdir(parents=True, exist_ok=False)
    alias_file = REPORT_DIR / "label_aliases.json"
    alias_file.write_text(json.dumps(LABEL_ALIASES, indent=2), encoding="utf-8")
    installed = json.loads((ROOT / "dependencies/conceptgraphs-runtime/installed.json").read_text())
    linux_python = installed["python"]
    def linux_path(path):
        return subprocess.check_output(["wsl.exe", "-d", WSL_DISTRO, "--exec", "wslpath", "-u", str(path)], text=True).strip()
    command = ["wsl.exe", "-d", WSL_DISTRO, "--exec", linux_python, "-u",
               linux_path(ROOT / "code/midterm_report/conceptgraphs_support/results_report.py"),
               "--project-root", linux_path(ROOT), "--experiment", EXPERIMENT_NAME, "--scene", SCENE,
               "--output", linux_path(REPORT_DIR), "--pixel-stride", str(PIXEL_STRIDE),
               "--voxel-size", str(VOXEL_SIZE_M), "--max-depth", str(MAX_DEPTH_M),
               "--semantic-threshold", str(SEMANTIC_ASSOCIATION_M), "--aliases", linux_path(alias_file),
               "--thresholds", *map(str, GEOMETRY_THRESHOLDS_M), "--excluded", *EXCLUDED_CLASSES]
    cg._run_logged(command, REPORT_DIR / "evaluation.log")
else:
    if REPORT_DIR is None:
        reports = list(REPORT_BASE.glob("report-*/report.json"))
        if not reports:
            raise FileNotFoundError("No completed report found; set RUN_EVALUATION=True.")
        REPORT_DIR = max(reports, key=lambda p: p.stat().st_mtime_ns).parent
    REPORT_DIR = Path(REPORT_DIR)

report = json.loads((REPORT_DIR / "report.json").read_text())
print("Report:", REPORT_DIR)
print("Scene:", report["scene"], "| frames:", len(report["frames"]))
print("Reference:", report["reference_kind"])
display(pd.read_csv(REPORT_DIR / "run_status.csv"))


## Constructed scenes: labeled nodes and real RGB evidence

This view links each retained graph node to the **actual RGB observations used for naming**, with detector boxes and model-predicted labels. `G` is its graph ID after pruning; `M` is its persistent original map ID. Colors track map objects across conditions. These are predicted node names, not dataset reference labels.

The image-backed graph uses a layout for readability. Use the separate 3D view for actual spatial positions. Scene photos show nodes with saved naming observations in that frame; they do not imply that every visible object was detected. The complete HTML gallery includes all retained nodes and up to `NODE_FRAME_LIMIT` complementary scene frames per condition.

New reports archive their underlying graph/prediction records. Earlier reports whose original graph outputs were deleted can still show saved figures, but missing node crops require a new report after graph construction finishes.

Yellow lines now show actual saved graph edges when both nodes have boxes in that frame. Arrows point from the subject to the supporting/containing object for on/in; nearby and next-to are undirected. Each frame reports its visible-edge count. The gallery starts with a frame containing edges when one is available; a frame can legitimately have zero edges. Relationship evidence views supplement naming views so saved co-visible pairs can be displayed.


In [ ]:
available_variants = [s["variant"] for s in report["sources"]]
if DISPLAY_VARIANT not in available_variants:
    DISPLAY_VARIANT = available_variants[0]
node_info_file = REPORT_DIR / "node_images.json"
if not node_info_file.is_file():
    node_info = ie.add_image_evidence(REPORT_DIR, ROOT, max_frames=NODE_FRAME_LIMIT)
else:
    node_info = json.loads(node_info_file.read_text())
    if node_info.get("max_frames_per_variant") != NODE_FRAME_LIMIT or node_info.get("render_version", 0) < 2:
        node_info = ie.add_image_evidence(REPORT_DIR, ROOT, max_frames=NODE_FRAME_LIMIT)
if node_info.get("missing_source_evidence"):
    print("Raw node evidence is unavailable for:", ", ".join(node_info["missing_source_evidence"]))
    print("Generate a new report after the clean graph run finishes; existing report images remain available.")
if node_info["variants"]:
    display(pd.DataFrame(node_info["variants"]))
    image_graph = REPORT_DIR / f"image_graph_{DISPLAY_VARIANT}.png"
    if image_graph.is_file():
        display(Image(filename=str(image_graph)))
    frames = pd.read_csv(REPORT_DIR / "scene_frame_index.csv")
    for _, row in frames[frames.variant == DISPLAY_VARIANT].head(3).iterrows():
        print("Scene frame", row["frame_id"], "| labeled graph nodes:", row["visible_graph_nodes"], "| saved edges visible:", row["visible_saved_edges"], "/", row["total_saved_edges"])
        display(Image(filename=str(REPORT_DIR / row["image"])))
    node_images = pd.read_csv(REPORT_DIR / "node_image_index.csv")
    selected_nodes = node_images[node_images.variant == DISPLAY_VARIANT]
    display(selected_nodes[["graph_id", "original_map_id", "predicted_label", "frame_id", "vote_agreement"]])
    cards = []
    for _, row in selected_nodes.head(NODE_CROP_PREVIEWS).iterrows():
        if not isinstance(row["crop"], str) or not row["crop"]:
            continue
        encoded = base64.b64encode((REPORT_DIR / row["crop"]).read_bytes()).decode("ascii")
        label = html.escape(str(row["predicted_label"]))
        cards.append(f'<div style="padding:12px;background:#f4f6f9"><b>G{row["graph_id"]} / M{row["original_map_id"]}: {label}</b><br><img style="max-width:240px;max-height:180px" src="data:image/jpeg;base64,{encoded}"><br>Frame {row["frame_id"]}</div>')
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:12px">' + ''.join(cards) + '</div>'))
    print("Complete image-grounded scene gallery:", REPORT_DIR / "scene_nodes.html")


## 3. Semantic labeling results

`shared_map` assigns each model condition's object names to the **same reconstructed geometry**, isolating naming differences. `retained_graph` includes the effects of node pruning; removed geometry becomes uncovered reference points. Report both scopes explicitly.

All percentages below are relative to the evaluated observed foreground. Review `unmapped_labels.csv` to distinguish vocabulary mismatch from missing geometric coverage. No labels are inferred from GT to improve predictions.


In [ ]:
semantics = pd.read_csv(REPORT_DIR / "semantic_metrics.csv")
semantic_display = semantics.copy()
percentage_columns = ["mAcc", "mIoU", "f_mIoU", "point_accuracy", "unknown_or_uncovered_rate", "geometric_coverage"]
semantic_display[percentage_columns] *= 100
semantic_display = semantic_display.rename(columns={c: c + " (%)" for c in percentage_columns})
display(semantic_display.round(2))
unknown_file = REPORT_DIR / "unmapped_labels.csv"
if unknown_file.is_file():
    print("Prediction labels without a declared dataset-category match:")
    display(pd.read_csv(unknown_file))


## 4. Geometric comparison with the observed GT reference

The shared-map row is common to all four VLM conditions. Retained-graph rows show how each condition's pruning changes surface coverage. Lower distance is better; higher precision, recall and F-score is better. Coverage is limited to the observed reference surfaces and declared class exclusions.


In [ ]:
geometry = pd.read_csv(REPORT_DIR / "geometry_metrics.csv")
geometry_display = geometry.copy()
geometry_percentages = [c for c in geometry if c.startswith(("precision_", "recall_", "fscore_"))]
geometry_display[geometry_percentages] *= 100
display(geometry_display.rename(columns={c: c + " (%)" for c in geometry_percentages}).round(4))


## 5. Per-class semantic results

Class support is measured in reference voxels, not object-instance counts. A large class can dominate f-mIoU, while mAcc gives each GT-supported class equal weight. The table contains all evaluated classes; the figure shows the most frequent classes for readability.


In [ ]:
per_class = pd.read_csv(REPORT_DIR / "per_class_metrics.csv")
variants = sorted(semantics["variant"].unique())
if DISPLAY_VARIANT not in variants:
    DISPLAY_VARIANT = variants[0]
selected = per_class[(per_class.variant == DISPLAY_VARIANT) & (per_class.scope == "shared_map")].sort_values("support", ascending=False)
display(selected.round(4))
plot_rows = selected[selected.support > 0].head(15).iloc[::-1]
ax = plot_rows.set_index("class_name")[["recall", "IoU"]].mul(100).plot.barh(figsize=(10, 7))
ax.set(xlabel="Percent", title=f"Observed semantic map: {DISPLAY_VARIANT}", xlim=(0, 100))
plt.tight_layout()
plt.savefig(REPORT_DIR / f"per_class_{DISPLAY_VARIANT}.png", dpi=160)
plt.show()


## 6. See the reference map and the proposed maps

The same category ID has the same color in both semantic views. Gray predicted points indicate unmapped labels. Visualizations include the whole observed reference for context, including background classes excluded from the quantitative foreground scores.

The interactive viewer is an offline HTML artifact: drag to rotate, scroll to zoom, and switch model conditions. The two PLY files can also be opened in an external point-cloud viewer.


In [ ]:
viewer_file = REPORT_DIR / "map_viewer.html"
viewer_html = viewer_file.read_text(encoding="utf-8")
display(HTML('<iframe style="width:100%;height:700px;border:0" srcdoc="' + html.escape(viewer_html, quote=True) + '"></iframe>'))
print("Standalone interactive viewer:", viewer_file)
display(Image(filename=str(REPORT_DIR / f"maps_{DISPLAY_VARIANT}.png")))
print("GT reference PLY:", REPORT_DIR / "observed_gt_semantic.ply")
print("Proposed map PLY:", REPORT_DIR / "proposed_shared_rgb.ply")


## 7. Inspect a GT semantic frame

This is the original categorical ground-truth mask paired with its RGB image. Numeric IDs are interpreted using this scene's own category dictionary. The 3D reference above fuses these observations using depth and camera poses.


In [ ]:
from PIL import Image as PILImage
PREVIEW_FRAME_ID = report["frames"][0]
scene_root = ROOT / "dataset/osma-bench/data" / SCENE
with PILImage.open(scene_root / "results" / f"frame{PREVIEW_FRAME_ID:06d}.jpg") as image:
    rgb = np.asarray(image.convert("RGB"))
with PILImage.open(scene_root / "results" / f"semantic{PREVIEW_FRAME_ID:06d}.png") as image:
    labels = np.asarray(image)
semantic_rgb = palette(labels.reshape(-1)).reshape(*labels.shape, 3)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(rgb); axes[0].set_title(f"RGB frame {PREVIEW_FRAME_ID}")
axes[1].imshow(semantic_rgb); axes[1].set_title("GT semantic categories")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
vocabulary = pd.read_csv(REPORT_DIR / "class_vocabulary.csv")
display(vocabulary[vocabulary.class_id.isin(np.unique(labels))])


## 8. Graph construction and inference diagnostics

Graph size, retention, resolved-pair rate, invalid outputs and view disagreement are diagnostics. They are not semantic correctness scores. A valid `none` answer counts as resolved, while a missing shared view is unresolved. Generation latency excludes model loading and map construction.


In [ ]:
diagnostics = pd.read_csv(REPORT_DIR / "graph_diagnostics.csv")
display(diagnostics.round(4))


## 9. Full graph overview

Positions come from saved 3D bounding-box centers. Edge labels preserve the upstream A/B relationship convention relative to source/target node IDs. The graph does not imply that all physically meaningful pairs were proposed.


In [ ]:
source = next(s for s in report["sources"] if s["variant"] == DISPLAY_VARIANT)
scene_dir = ROOT / "artifacts/scene_graphs" / EXPERIMENT_NAME / SCENE.replace("/", "__")
variant_dir = ie.resolve_evidence(REPORT_DIR, source, scene_dir)
if variant_dir is None:
    saved_graph = REPORT_DIR / f"graph_{DISPLAY_VARIANT}.png"
    if saved_graph.is_file():
        display(Image(filename=str(saved_graph)))
    print("Original graph records were cleared. New reports archive graph evidence so this view survives cleanup.")
else:
    graph = json.loads((variant_dir / "graph.json").read_text())
    positions = {n["id"]: np.asarray(n["bbox_center"], float) for n in graph["nodes"]}
    fig = plt.figure(figsize=(12, 8)); ax = fig.add_subplot(projection="3d")
    for node in graph["nodes"]:
        point = positions[node["id"]]
        ax.scatter(*point, s=20)
        ax.text(*point, f"{node['id']}: {node['object_tag']}", fontsize=7)
    for edge in graph["edges"]:
        a, b = positions[edge["source"]], positions[edge["target"]]
        ax.plot(*np.stack([a, b]).T, color="gray", linewidth=1)
        ax.text(*((a+b)/2), f"{edge['source']} / {edge['target']}: {edge['relation']}", fontsize=7, color="darkred")
    ax.set(xlabel="X (m)", ylabel="Y (m)", zlabel="Z (m)", title=DISPLAY_VARIANT)
    ax.view_init(elev=20, azim=-60, vertical_axis="y")
    plt.tight_layout(); plt.savefig(REPORT_DIR / f"graph_{DISPLAY_VARIANT}.png", dpi=160); plt.show()


## 10. Successful and failed relationship cases

The report includes actual shared RGB views with A/B boxes and per-view outputs. `FAIL` cases can demonstrate invalid formatting, tied votes, missing evidence or upstream rejection. A valid edge is only a **candidate success** until independently reviewed. Reviewed relation matches establish relationship success for that pair, not complete correctness of its object map or whole graph.

Fill `reviewed_relation`, `failure_stage`, and `notes` in the selected graph run's `review.csv`, then rebuild a report to incorporate those reviews. Use canonical labels such as `a_on_b`, `b_on_a`, `a_in_b`, `b_in_a`, `next_to`, `nearby`, or `none`. Review a fixed set of original map IDs across models; include meaningful omitted pairs if reporting graph recall. No relationship accuracy/F1 is fabricated from the models' own answers.


In [ ]:
cases = pd.read_csv(REPORT_DIR / "qualitative_cases.csv")
display(cases)
for status in cases["status"].unique():
    print(status)
    for _, row in cases[cases.status == status].head(2).iterrows():
        display(Image(filename=str(REPORT_DIR / row["image"])))
print("Complete case gallery:", REPORT_DIR / "qualitative_cases.html")


## Interpretation and reproducibility

`report.json` records exact graph run folders, frame IDs, map metadata, mapping fingerprint, evaluator hash, exclusions, alias choices, sampling resolution and thresholds. Failed/partial graph attempts are listed in `run_status.csv` and excluded from completed-result comparisons. Rebuilding creates a new report directory.

Report these results as **observed-frame semantic map evaluation**. True object-instance AP, complete-room reconstruction accuracy and complete ground-truth scene-graph precision/recall require additional instance/geometry/edge annotations. Current nearest-neighbor foreground scores, observed-frame coverage and reviewed relationship case studies are separate measurements with different denominators.
